
# Regresión logística
### Material introductorio para alumnos de Ingeniería Mecatrónica

---

## Objetivo
Comprender de manera sencilla qué es la **regresión logística**, para qué sirve y cómo puede aplicarse en problemas básicos de **clasificación** dentro de Ingeniería Mecatrónica.

## ¿Qué aprenderás?
Al finalizar este notebook podrás:

- distinguir entre un problema de **regresión** y uno de **clasificación**,
- entender la idea de probabilidad en clasificación binaria,
- aplicar regresión logística con Python,
- interpretar predicciones y probabilidades,
- evaluar un modelo de forma básica.

---

## Contexto en Mecatrónica
En Ingeniería Mecatrónica aparecen con frecuencia problemas donde la salida no es un número continuo, sino una **clase** o **categoría**, por ejemplo:

- **pieza aceptada / pieza defectuosa**,
- **motor estable / motor inestable**,
- **sensor funcionando / sensor con falla**,
- **temperatura normal / temperatura peligrosa**,
- **mantenimiento no requerido / mantenimiento requerido**.

En esos casos, la **regresión logística** es una de las herramientas más sencillas para comenzar.



## 1. ¿Qué es la regresión logística?

Aunque su nombre contiene la palabra *regresión*, en realidad la regresión logística se usa principalmente para **clasificación**.

Su objetivo es estimar la **probabilidad** de que una observación pertenezca a una clase.

Por ejemplo:

- probabilidad de que una pieza sea defectuosa,
- probabilidad de que un motor requiera mantenimiento,
- probabilidad de que una lectura corresponda a una condición de alarma.

### Clasificación binaria
En su forma más básica, la regresión logística trabaja con dos clases:

- **0** = No
- **1** = Sí

Ejemplo:

- 0 = pieza correcta
- 1 = pieza defectuosa



## 2. Diferencia entre regresión lineal y regresión logística

### Regresión lineal
Se usa cuando queremos predecir un valor continuo, como:

- temperatura,
- velocidad,
- corriente,
- posición.

### Regresión logística
Se usa cuando queremos predecir una clase, como:

- aprobado / rechazado,
- normal / anormal,
- falla / sin falla.

La salida del modelo logístico es una **probabilidad entre 0 y 1**.



## 3. Idea básica del modelo

La regresión logística utiliza una función llamada **sigmoide**, que transforma cualquier valor en un número entre 0 y 1:

\[
P(y=1) = \frac{1}{1 + e^{-(mx+b)}}
\]

Esto permite interpretar el resultado como probabilidad.

### Interpretación sencilla
- Si la probabilidad es cercana a **0**, el modelo cree que pertenece a la clase 0.
- Si la probabilidad es cercana a **1**, el modelo cree que pertenece a la clase 1.
- Normalmente se usa un umbral de **0.5**:
  - probabilidad < 0.5 → clase 0
  - probabilidad ≥ 0.5 → clase 1


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.precision", 3)



## 4. Ejemplo sencillo: vibración de un motor y condición de falla

Supongamos que medimos el nivel de vibración de un motor en mm/s y queremos clasificar si el motor presenta una condición de falla:

- **0** = funcionamiento normal
- **1** = posible falla

Los datos siguientes son ficticios, pero didácticamente razonables.


In [ ]:

vibracion = np.array([0.8, 1.0, 1.2, 1.5, 1.7, 2.0, 2.2, 2.5, 2.9, 3.1, 3.4, 3.8, 4.1, 4.5])
falla =     np.array([0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1,   1,   1])

df = pd.DataFrame({
    "Vibracion_mm_s": vibracion,
    "Falla": falla
})

df



## 5. Visualización inicial

Graficamos los datos para observar el patrón.


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(vibracion, falla)
plt.title("Vibración del motor vs condición de falla")
plt.xlabel("Vibración (mm/s)")
plt.ylabel("Clase (0 = normal, 1 = falla)")
plt.grid(True)
plt.show()



Observamos que para niveles bajos de vibración la clase suele ser **0**, y para niveles altos la clase suele ser **1**.

Esto es un problema típico de clasificación binaria.



## 6. Entrenamiento del modelo de regresión logística

En `scikit-learn`, el modelo se implementa con `LogisticRegression`.


In [ ]:

X = vibracion.reshape(-1, 1)   # entrada
y = falla                     # salida

modelo = LogisticRegression()
modelo.fit(X, y)

print("Coeficiente:", modelo.coef_[0][0])
print("Intercepto:", modelo.intercept_[0])



## 7. Probabilidades estimadas

El modelo no solo predice 0 o 1. También puede estimar la **probabilidad** de pertenecer a cada clase.


In [ ]:

probabilidades = modelo.predict_proba(X)

tabla_prob = pd.DataFrame({
    "Vibracion_mm_s": vibracion,
    "Prob_clase_0": probabilidades[:, 0],
    "Prob_clase_1": probabilidades[:, 1],
    "Clase_predicha": modelo.predict(X)
})

tabla_prob



La columna `Prob_clase_1` indica la probabilidad de que el sistema pertenezca a la clase **falla**.



## 8. Visualización de la curva logística

Ahora graficamos una curva suave para ver cómo cambia la probabilidad.


In [ ]:

x_suave = np.linspace(vibracion.min()-0.5, vibracion.max()+0.5, 200).reshape(-1, 1)
prob_suave = modelo.predict_proba(x_suave)[:, 1]

plt.figure(figsize=(8,5))
plt.scatter(vibracion, falla, label="Datos reales")
plt.plot(x_suave, prob_suave, label="Probabilidad de falla")
plt.axhline(0.5, linestyle="--", label="Umbral 0.5")
plt.title("Curva de regresión logística")
plt.xlabel("Vibración (mm/s)")
plt.ylabel("Probabilidad / clase")
plt.grid(True)
plt.legend()
plt.show()



## 9. Predicción para un nuevo valor

Supongamos que ahora medimos una vibración de **2.3 mm/s**.  
Queremos saber la probabilidad de falla y la clase estimada.


In [ ]:

nueva_vibracion = np.array([[2.3]])

prob_nueva = modelo.predict_proba(nueva_vibracion)[0, 1]
clase_nueva = modelo.predict(nueva_vibracion)[0]

print(f"Probabilidad de falla: {prob_nueva:.4f}")
print(f"Clase predicha: {clase_nueva}")



## 10. Interpretación del resultado

- Si la probabilidad es alta, el sistema tiene mayor posibilidad de pertenecer a la clase **falla**.
- Si la probabilidad es baja, se clasifica como **normal**.

Esto es útil para sistemas de monitoreo, mantenimiento predictivo y diagnóstico básico.



## 11. Frontera de decisión

La frontera de decisión es el punto donde el modelo cambia de una clase a otra, usando normalmente un umbral de 0.5.

Podemos explorar qué valores producen clase 0 y cuáles producen clase 1.


In [ ]:

valores_prueba = np.array([[1.4], [1.9], [2.1], [2.4], [3.0], [4.0]])
predicciones_prueba = modelo.predict(valores_prueba)
probabilidades_prueba = modelo.predict_proba(valores_prueba)[:, 1]

pd.DataFrame({
    "Vibracion_mm_s": valores_prueba.flatten(),
    "Probabilidad_falla": probabilidades_prueba,
    "Clase_predicha": predicciones_prueba
})



## 12. Evaluación básica del modelo

Podemos comparar las clases reales con las clases predichas.


In [ ]:

y_pred = modelo.predict(X)

exactitud = accuracy_score(y, y_pred)
matriz = confusion_matrix(y, y_pred)

print(f"Accuracy: {exactitud:.4f}")
print("\nMatriz de confusión:")
print(matriz)



### Interpretación de la matriz de confusión

La matriz de confusión indica:

- cuántos casos de clase 0 fueron clasificados correctamente,
- cuántos casos de clase 1 fueron clasificados correctamente,
- cuántos errores cometió el modelo.

En problemas reales, esto ayuda a valorar si el sistema es confiable.


In [ ]:

print(classification_report(y, y_pred, zero_division=0))



## 13. Ejemplo aplicado a inspección de piezas

Supongamos ahora un caso simple de inspección donde una pieza se clasifica con base en la temperatura detectada durante una prueba:

- 0 = pieza aceptada
- 1 = pieza con posible defecto térmico


In [ ]:

temperatura = np.array([28, 30, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49], dtype=float)
defecto =     np.array([0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1], dtype=int)

X2 = temperatura.reshape(-1, 1)
y2 = defecto

modelo2 = LogisticRegression()
modelo2.fit(X2, y2)

x2_suave = np.linspace(27, 50, 200).reshape(-1, 1)
p2 = modelo2.predict_proba(x2_suave)[:, 1]

plt.figure(figsize=(8,5))
plt.scatter(temperatura, defecto, label="Datos")
plt.plot(x2_suave, p2, label="Probabilidad de defecto")
plt.axhline(0.5, linestyle="--", label="Umbral 0.5")
plt.title("Temperatura vs defecto de pieza")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Probabilidad / clase")
plt.grid(True)
plt.legend()
plt.show()



## 14. Ventajas de la regresión logística

- Es sencilla de entender e implementar.
- Funciona bien en problemas de clasificación binaria.
- Entrega probabilidades, no solo clases.
- Es una buena base para comenzar en Machine Learning.

## 15. Limitaciones

- No siempre separa bien clases complejas.
- Puede fallar si la relación entre variables y clases es muy no lineal.
- Puede verse afectada por datos mal balanceados o por variables poco informativas.
- En problemas reales, a veces se necesitan más variables o modelos más potentes.



## 16. Conclusión

La regresión logística es una herramienta básica y muy útil para resolver problemas de **clasificación binaria**.

En Ingeniería Mecatrónica puede usarse para tareas como:

- detección de fallas,
- clasificación de estados de operación,
- inspección de calidad,
- apoyo al mantenimiento predictivo.

Además, permite dar el primer paso hacia modelos más avanzados de clasificación.



# Ejercicios propuestos

## Ejercicio 1. Comprensión conceptual
Responde con tus propias palabras:

1. ¿Por qué la regresión logística se usa para clasificación y no para predecir valores continuos?
2. ¿Qué representa una probabilidad de 0.85?
3. ¿Qué significa usar un umbral de 0.5?

---

## Ejercicio 2. Sensor de temperatura
Crea un conjunto de datos donde:

- la entrada sea temperatura,
- la salida sea `0 = normal` y `1 = alarma`.

Después:

1. organiza los datos en un DataFrame,
2. entrena un modelo de regresión logística,
3. grafica la curva logística,
4. estima la probabilidad de alarma para un nuevo valor.

---

## Ejercicio 3. Inspección de piezas
Supón que una cámara mide el ancho de una pieza y deseas clasificarla como:

- `0 = aceptada`
- `1 = defectuosa`

Usa datos de ejemplo, entrena el modelo y responde:

1. ¿qué ancho parece marcar el cambio entre clases?
2. ¿qué probabilidad obtiene una pieza intermedia?
3. ¿cómo interpretarías el resultado en un sistema automatizado?

---

## Ejercicio 4. Evaluación
Con un conjunto de datos sencillo:

1. entrena el modelo,
2. obtén las predicciones,
3. calcula `accuracy`,
4. genera la matriz de confusión,
5. interpreta los errores.

---

## Ejercicio 5. Reflexión aplicada
Menciona un caso real de Ingeniería Mecatrónica donde usarías regresión logística y explica:

- cuál sería la variable de entrada,
- qué significan las clases 0 y 1,
- por qué esta técnica sería útil.



# Actividad integradora sugerida

## Mini proyecto
Diseña un pequeño problema de clasificación binaria relacionado con mecatrónica, por ejemplo:

- motor con falla / sin falla,
- pieza aceptada / defectuosa,
- temperatura segura / peligrosa,
- sistema estable / inestable.

Entrega:

1. tabla de datos,
2. gráfica de los datos,
3. entrenamiento del modelo,
4. probabilidades estimadas,
5. gráfica de la curva logística,
6. predicción de un caso nuevo,
7. conclusión técnica breve.
